In [3]:

import pandas as pd
import numpy as np
from collections import Counter, defaultdict



# 1. LOAD DATASET
df = pd.read_csv(r"C:\Users\Karishma\anaconda3\NLP Skill\small_ud_ewt_pos_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nFirst 10 rows:")
print(df.head(10))



# 2. EXTRACT WORDS AND POS TAGS

sentences = []

for sentence_id, group in df.groupby("sentence_id"):
    
    sentence = list(
        zip(
            group["word"],
            group["pos"]
        )
    )
    
    sentences.append(sentence)


print("\nTotal sentences:", len(sentences))

print("\nFirst sentence:")
print(sentences[0])



# 3. TRAIN-TEST SPLIT

split = int(len(sentences) * 0.80)

train_sentences = sentences[:split]
test_sentences = sentences[split:]

print("\nTraining sentences:", len(train_sentences))
print("Testing sentences :", len(test_sentences))



# 4. CALCULATE TRANSITION COUNTS


transition_counts = defaultdict(Counter)

tag_counts = Counter()

START = "<START>"
END = "<END>"

for sentence in train_sentences:
    
    previous_tag = START
    
    for word, tag in sentence:
        
        transition_counts[previous_tag][tag] += 1
        
        tag_counts[tag] += 1
        
        previous_tag = tag
    
    transition_counts[previous_tag][END] += 1


# Get all POS tags
tags = sorted(tag_counts.keys())

number_of_tags = len(tags)

print("\nPOS Tags:")
print(tags)



# 5. CALCULATE TRANSITION PROBABILITIES

transition_probabilities = defaultdict(dict)

for previous_tag in transition_counts:
    
    total = sum(
        transition_counts[previous_tag].values()
    )
    
    for current_tag in transition_counts[previous_tag]:
        
        probability = (
            transition_counts[previous_tag][current_tag]
            / total
        )
        
        transition_probabilities[
            previous_tag
        ][current_tag] = probability


print("\nSample Transition Probabilities:")

for previous_tag in list(
    transition_probabilities.keys()
)[:5]:
    
    print(
        previous_tag,
        "->",
        transition_probabilities[previous_tag]
    )



# 6. CALCULATE EMISSION COUNTS


emission_counts = defaultdict(Counter)

vocabulary = set()

for sentence in train_sentences:
    
    for word, tag in sentence:
        
        word = word.lower()
        
        emission_counts[tag][word] += 1
        
        vocabulary.add(word)


print("\nVocabulary size:", len(vocabulary))


# 7. CALCULATE EMISSION PROBABILITIES

emission_probabilities = defaultdict(dict)

for tag in emission_counts:
    
    total = tag_counts[tag]
    
    for word in emission_counts[tag]:
        
        probability = (
            emission_counts[tag][word]
            / total
        )
        
        emission_probabilities[tag][word] = probability


print("\nSample Emission Probabilities:")

for tag in list(
    emission_probabilities.keys()
)[:5]:
    
    print(
        tag,
        "->",
        list(
            emission_probabilities[tag].items()
        )[:5]
    )


# 8. LOG PROBABILITY WITH LAPLACE SMOOTHING

def log_transition(previous_tag, current_tag):
    
    count = transition_counts[
        previous_tag
    ][current_tag]
    
    total = sum(
        transition_counts[
            previous_tag
        ].values()
    )
    
    probability = (
        count + 1
    ) / (
        total + number_of_tags + 1
    )
    
    return np.log(probability)


def log_emission(tag, word):
    
    word = word.lower()
    
    count = emission_counts[tag][word]
    
    total = tag_counts[tag]
    
    probability = (
        count + 1
    ) / (
        total + len(vocabulary) + 1
    )
    
    return np.log(probability)


# 9. VITERBI ALGORITHM

def viterbi(words):
    
    number_of_words = len(words)
    
    if number_of_words == 0:
        return []
    
    
    # Dynamic programming table
    dp = np.full(
        (
            number_of_words,
            number_of_tags
        ),
        -np.inf
    )
    
    
    # Backpointer table
    backpointer = np.zeros(
        (
            number_of_words,
            number_of_tags
        ),
        dtype=int
    )
    
    # First word
   
    for j, tag in enumerate(tags):
        
        dp[0][j] = (
            log_transition(
                START,
                tag
            )
            +
            log_emission(
                tag,
                words[0]
            )
        )
    
    # Remaining words
    
    for i in range(
        1,
        number_of_words
    ):
        
        for j, current_tag in enumerate(tags):
            
            scores = []
            
            for k, previous_tag in enumerate(tags):
                
                score = (
                    dp[i - 1][k]
                    +
                    log_transition(
                        previous_tag,
                        current_tag
                    )
                )
                
                scores.append(score)
            
            
            best_previous = np.argmax(
                scores
            )
            
            
            dp[i][j] = (
                scores[best_previous]
                +
                log_emission(
                    current_tag,
                    words[i]
                )
            )
            
            
            backpointer[i][j] = (
                best_previous
            )
    
    # Find best final tag
   
    final_scores = []
    
    for j, tag in enumerate(tags):
        
        score = (
            dp[
                number_of_words - 1
            ][j]
            +
            log_transition(
                tag,
                END
            )
        )
        
        final_scores.append(score)
    
    
    best_last = np.argmax(
        final_scores
    )
    
    # Backtracking
    
    best_path = [best_last]
    
    
    for i in range(
        number_of_words - 1,
        0,
        -1
    ):
        
        best_path.append(
            backpointer[
                i
            ][
                best_path[-1]
            ]
        )
    
    
    best_path.reverse()
    
    
    predicted_tags = [
        tags[index]
        for index in best_path
    ]
    
    
    return predicted_tags

# 10. TEST VITERBI ALGORITHM

test_sentence = [
    "The",
    "student",
    "reads",
    "a",
    "book",
    "."
]

predicted_tags = viterbi(
    test_sentence
)

print("\nVITERBI TEST")
print("=" * 40)

for word, tag in zip(
    test_sentence,
    predicted_tags
):
    
    print(
        word,
        "->",
        tag
    )



# 11. POS TAGGING FUNCTION

def tag_sentence(sentence):
    
    words = sentence.strip().split()
    
    cleaned_words = []
    
    
    for word in words:
        
        # Handle punctuation at the end
        if (
            word.endswith(".")
            and word != "."
        ):
            
            cleaned_words.append(
                word[:-1]
            )
            
            cleaned_words.append(".")
        
        else:
            
            cleaned_words.append(
                word
            )
    
    
    predicted_tags = viterbi(
        cleaned_words
    )
    
    
    return list(
        zip(
            cleaned_words,
            predicted_tags
        )
    )


# 12. REQUIRED SAMPLE INPUT

sample_sentence = (
    "The student reads a book."
)

result = tag_sentence(
    sample_sentence
)

print("\nSAMPLE INPUT")
print("=" * 40)

print("Input:")
print(sample_sentence)

print("\nOutput:")

for word, tag in result:
    
    print(
        f"{word} -> {tag}"
    )


# 13. EVALUATE ON TEST DATASET

correct = 0
total = 0


for sentence in test_sentences:
    
    words = [
        word
        for word, tag in sentence
    ]
    
    actual_tags = [
        tag
        for word, tag in sentence
    ]
    
    
    predicted_tags = viterbi(
        words
    )
    
    
    for actual, predicted in zip(
        actual_tags,
        predicted_tags
    ):
        
        if actual == predicted:
            correct += 1
        
        total += 1


accuracy = (
    correct / total
    if total > 0
    else 0
)


# 14. DISPLAY ACCURACY


print("\nTEST DATASET RESULTS")
print("=" * 40)

print(
    "Correct predictions:",
    correct
)

print(
    "Total POS tags:",
    total
)

print(
    f"POS Tagging Accuracy: "
    f"{accuracy * 100:.2f}%"
)


# 15. EVALUATION REPORT

report = defaultdict(
    lambda: {
        "correct": 0,
        "predicted": 0,
        "actual": 0
    }
)


for sentence in test_sentences:
    
    words = [
        word
        for word, tag in sentence
    ]
    
    actual_tags = [
        tag
        for word, tag in sentence
    ]
    
    
    predicted_tags = viterbi(
        words
    )
    
    
    for actual, predicted in zip(
        actual_tags,
        predicted_tags
    ):
        
        report[actual]["actual"] += 1
        
        report[predicted]["predicted"] += 1
        
        
        if actual == predicted:
            
            report[actual]["correct"] += 1


# 16. PRECISION, RECALL AND F1-SCORE

print("\nPOS TAG EVALUATION REPORT")
print("=" * 65)

print(
    f"{'TAG':<10}"
    f"{'PRECISION':<15}"
    f"{'RECALL':<15}"
    f"{'F1-SCORE':<15}"
)

print("-" * 65)


for tag in sorted(report):
    
    correct_count = (
        report[tag]["correct"]
    )
    
    predicted_count = (
        report[tag]["predicted"]
    )
    
    actual_count = (
        report[tag]["actual"]
    )
    
    
    # Precision
    if predicted_count > 0:
        
        precision = (
            correct_count
            / predicted_count
        )
    
    else:
        
        precision = 0
    
    
    # Recall
    if actual_count > 0:
        
        recall = (
            correct_count
            / actual_count
        )
    
    else:
        
        recall = 0
    
    
    # F1 Score
    if (
        precision + recall
        > 0
    ):
        
        f1_score = (
            2
            * precision
            * recall
            / (
                precision
                + recall
            )
        )
    
    else:
        
        f1_score = 0
    
    
    print(
        f"{tag:<10}"
        f"{precision:<15.2f}"
        f"{recall:<15.2f}"
        f"{f1_score:<15.2f}"
    )


# 17. USER INPUT

print("\nCUSTOM SENTENCE POS TAGGING")
print("=" * 40)

user_sentence = input(
    "Enter a sentence: "
)


if user_sentence.strip():
    
    result = tag_sentence(
        user_sentence
    )
    
    print("\nPredicted POS Tags:")
    
    
    for word, tag in result:
        
        print(
            f"{word} -> {tag}"
        )

else:
    
    print(
        "No sentence entered."
    )









Dataset loaded successfully!
Dataset shape: (116, 3)

First 10 rows:
   sentence_id     word    pos
0            1      The    DET
1            1  student   NOUN
2            1    reads   VERB
3            1        a    DET
4            1     book   NOUN
5            1        .  PUNCT
6            2      The    DET
7            2  teacher   NOUN
8            2   writes   VERB
9            2        a    DET

Total sentences: 20

First sentence:
[('The', 'DET'), ('student', 'NOUN'), ('reads', 'VERB'), ('a', 'DET'), ('book', 'NOUN'), ('.', 'PUNCT')]

Training sentences: 16
Testing sentences : 4

POS Tags:
['ADP', 'ADV', 'DET', 'NOUN', 'PROPN', 'PUNCT', 'VERB']

Sample Transition Probabilities:
<START> -> {'DET': 1.0}
DET -> {'NOUN': 1.0}
NOUN -> {'VERB': 0.6153846153846154, 'PUNCT': 0.38461538461538464}
VERB -> {'DET': 0.4375, 'ADP': 0.1875, 'ADV': 0.3125, 'PROPN': 0.0625}
PUNCT -> {'<END>': 1.0}

Vocabulary size: 32

Sample Emission Probabilities:
DET -> [('the', 0.5), ('a', 0.4615384615

Enter a sentence:  I love nlp



Predicted POS Tags:
I -> DET
love -> NOUN
nlp -> PUNCT
